# KNPS 2026 해안쓰레기 검출 추론 (앙상블 + D4 TTA + 산식 맞춤 후처리)

- 입력: `AIF_INPUT_DIR` 아래 `patches.csv`(`id` 열)와 `images/<id>.png`
- 출력: `AIF_PREDICTION_PATH` 에 `id,rle` CSV
- 모델·후처리 설정: `assets/model/config.json` (tune.py 가 생성). 없으면 주최 측 베이스라인 가중치로 동작
- 추론 모듈: `assets/code/knps/` (package.py 가 복사). 작업 폴더에서 돌리면 `./knps` 를 씁니다
- 추론 단계는 네트워크 차단, `!`/`%` 줄은 제거되므로 모든 동작을 파이썬으로 작성했습니다

**로컬 실행**
```
AIF_INPUT_DIR=<입력 폴더> AIF_PREDICTION_PATH=./prediction.csv jupyter nbconvert --to notebook --execute predict.ipynb
```

In [ ]:
# [setup]
import csv
import io
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import torch
from PIL import Image

# 추론 모듈 위치: 제출 폴더는 assets/code/knps, 작업 폴더는 ./knps
for _p in ("assets/code", "."):
    if Path(_p, "knps", "model.py").exists():
        sys.path.insert(0, str(Path(_p).resolve()))
        break
from knps.model import SmpSegCls, init_from_baseline, load_checkpoint, predict_probs
from knps.postprocess import DEFAULT_CFG, postprocess
from knps.rle import rle_decode, rle_encode

INPUT_DIR = Path(os.environ.get("AIF_INPUT_DIR", "./input"))
PREDICTION_PATH = Path(os.environ.get("AIF_PREDICTION_PATH", "./prediction.csv"))
CONFIG = Path("assets/model/config.json")
BASELINE = Path("assets/model/unet_r18_debris_lite.pt")
H, W = 256, 256
print("knps from", sys.path[0])

In [ ]:
# [inputs]
# 대상 목록은 patches.csv 로만 정합니다 (images/ 폴더를 훑지 않음)
print("AIF_INPUT_DIR:", INPUT_DIR, sorted(p.name for p in INPUT_DIR.iterdir()) if INPUT_DIR.is_dir() else "(없음)")
lists = sorted(INPUT_DIR.rglob("patches.csv"))
if not lists:
    zs = sorted(INPUT_DIR.rglob("*.zip")) if INPUT_DIR.is_dir() else []
    if len(zs) == 1:
        import tempfile, zipfile
        dst = Path(tempfile.mkdtemp()) / "input"
        with zipfile.ZipFile(zs[0]) as z:
            z.extractall(dst)
        print(f"입력이 압축본이어서 풀어서 읽습니다: {zs[0].name} -> {dst}")
        lists = sorted(dst.rglob("patches.csv"))
if not lists:
    raise SystemExit(f"patches.csv 를 찾지 못했습니다: {INPUT_DIR}")
ROOT = lists[0].parent
with lists[0].open(encoding="utf-8-sig", newline="") as f:
    IDS = [row["id"].strip() for row in csv.DictReader(f) if row.get("id", "").strip()]
IDS = list(dict.fromkeys(IDS))  # 중복 id 는 채점 실패이므로 한 번만
print(f"입력 루트 {ROOT}, 패치 {len(IDS)}장")

In [ ]:
# [env]
print("python", sys.version.split()[0], "torch", torch.__version__)
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# [model]
if CONFIG.exists():
    CONF = json.load(open(CONFIG, encoding="utf-8"))
    CFG = {**DEFAULT_CFG, **CONF["cfg"]}
else:
    # 베이스라인 가중치: 분류 헤드가 학습되지 않았으므로 확률맵만으로 판정 (원본 argmax+50px 와 거의 같음)
    CONF = {"models": [], "tta": 8, "batch": 32}
    CFG = {**DEFAULT_CFG, "alpha": 0.0, "t_img": 0.5, "t_pix": 0.5, "rescue_px": 0}
TTA = int(CONF.get("tta", 8))
WEIGHTS = CONF.get("weights")   # 앙상블 모델별 가중치 (없으면 동일)
SCALES = tuple(CONF.get("tta_scales", [1.0]))   # 크기 바꾼 TTA (없으면 원본만)
BATCH = int(CONF.get("batch", 32))


def load_all(device):
    if CONF["models"]:
        return [load_checkpoint(p, device)[0] for p in CONF["models"]]
    m = SmpSegCls("unet", "resnet18", pretrained=False)
    init_from_baseline(m, BASELINE)
    return [m.to(device).eval()]


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
try:
    MODELS = load_all(DEVICE)
    predict_probs(MODELS, torch.zeros(1, 3, H, W, device=DEVICE), tta=1)
except Exception as e:  # noqa: BLE001
    print(f"{DEVICE} 에서 준비 실패({type(e).__name__}: {e}) → CPU 로 전환")
    DEVICE = "cpu"
    MODELS = load_all(DEVICE)
if DEVICE == "cpu" and TTA > 2:
    print(f"CPU 추론이라 시간 제한을 고려해 TTA {TTA} → 2 로 줄입니다")
    TTA = 2
print("device", DEVICE, "models", len(MODELS), "TTA", TTA, "cfg", CFG)

In [ ]:
# [infer-setup] 긴 추론은 아래 4개 셀로 나눠 셀마다 진행 기록이 남도록 합니다
ROWS = {}
T0 = time.time()


def read_image(path: Path) -> np.ndarray:
    with Image.open(io.BytesIO(path.read_bytes())) as im:
        arr = np.asarray(im.convert("RGB"))
    if arr.shape[:2] != (H, W):
        raise ValueError(f"{path.name}: 크기 {arr.shape[:2]}")
    return arr


def run_chunk(k: int, n_chunks: int = 4):
    lo, hi = len(IDS) * k // n_chunks, len(IDS) * (k + 1) // n_chunks
    for s in range(lo, hi, BATCH):
        ids = IDS[s:min(s + BATCH, hi)]
        x = np.stack([read_image(ROOT / "images" / f"{i}.png") for i in ids]).astype(np.float32) / 255.0
        prob, cls = predict_probs(MODELS, torch.from_numpy(x).permute(0, 3, 1, 2).to(DEVICE), tta=TTA, weights=WEIGHTS, scales=SCALES)
        prob, cls = prob.float().cpu().numpy(), cls.float().cpu().numpy()
        for i, p, c in zip(ids, prob, cls):
            ROWS[i] = rle_encode(postprocess(p, float(c), CFG))
    n_pos = sum(1 for r in ROWS.values() if r)
    print(f"  chunk {k + 1}/{n_chunks}: {len(ROWS)}/{len(IDS)} 완료, 양성 {n_pos}, {time.time() - T0:.1f}s", flush=True)

In [ ]:
# [infer-1]
run_chunk(0)

In [ ]:
# [infer-2]
run_chunk(1)

In [ ]:
# [infer-3]
run_chunk(2)

In [ ]:
# [infer-4]
run_chunk(3)

In [ ]:
# [save] 채점 실패 조건(누락·중복·잘못된 RLE)을 저장 전에 스스로 점검합니다
missing = [i for i in IDS if i not in ROWS]
assert not missing, f"예측 누락 {len(missing)}개"
for i in IDS:
    rle_decode(ROWS[i])
PREDICTION_PATH.parent.mkdir(parents=True, exist_ok=True)
with PREDICTION_PATH.open("w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id", "rle"])
    w.writerows((i, ROWS[i]) for i in IDS)
n_pos = sum(1 for i in IDS if ROWS[i])
print(f"저장했습니다: {PREDICTION_PATH} ({PREDICTION_PATH.stat().st_size} bytes), 양성 {n_pos}/{len(IDS)}, 총 {time.time() - T0:.1f}s")

## 제출

**반드시 `package.py` 로 만든 제출 폴더(코랩: `내 드라이브/sea_trash_data/submission`, 로컬: `C:\sea_trash_data\submission`) 안의 이 노트북에서 실행하세요.**
aifactory submit 은 노트북이 들어 있는 폴더 전체를 올리므로, 작업 폴더에서 실행하면 학습 데이터까지 올라갑니다.
처음 제출하는 구성은 `--debug` 로 먼저 확인하세요 (하루 10회, 점수 미반영).

참여키는 `C:\sea_trash_data\aifactory_key.txt` 에서 자동으로 읽습니다 (프로필 → 마이페이지 → 활동히스토리 탭에서 확인).
아래 셀의 `DEBUG = True` 로 먼저 디버그 제출하고, 확인되면 `False` 로 바꿔 본 제출하세요.

In [ ]:
# 참여키는 노트북에 적지 않고 OneDrive 밖의 파일에서 읽습니다 (git·제출 폴더·공유 노트북에 키가 남지 않게).
# 파일: C:\sea_trash_data\aifactory_key.txt (한 줄짜리). 다른 PC 에서는 환경변수 AIFACTORY_KEY 로도 됩니다.
# 채점 서버에서는 이 셀의 % 줄이 지워지고 아래 파이썬 줄만 실행되므로, 파일이 없어도 오류가 나지 않게 했습니다.
import os
from pathlib import Path

_key_file = Path(r"C:\sea_trash_data\aifactory_key.txt")
KEY = os.environ.get("AIFACTORY_KEY") or (_key_file.read_text(encoding="utf-8").strip() if _key_file.exists() else "")
DEBUG = True    # 처음 제출하는 구성은 True(디버그: 하루 10회, 점수 미반영)로 확인한 뒤 False 로 바꿔 본 제출
FLAG = "--debug" if DEBUG else ""
print("참여키:", (KEY[:4] + "…" + KEY[-4:]) if KEY else "없음 (C:\\sea_trash_data\\aifactory_key.txt 를 만드세요)",
      "| 디버그 제출" if DEBUG else "| 본 제출")
%pip install -q --upgrade aifactory
%load_ext aifactory
%aifactory submit --api-key {KEY} --model-name dbgB-v4cfgdil {FLAG}
